# Etapa 0: reconocimiento de la base SEPA

Exploración de un día de la base **Precios Claros - SEPA** (datos.produccion.gob.ar) para entender
qué publica, cómo viene el archivo y cuánto pesa, antes de escribir el pipeline.

Este notebook descarga el archivo de un día de la semana (si no está ya en `data/crudo/`), lo descomprime,
lo carga en DuckDB y responde:

1. Qué archivos hay, cada cuánto se actualizan y si las URLs son estables.
2. Cómo viene el paquete: compresión, tablas, separador, codificación, columnas.
3. Tamaños: comprimido, descomprimido, filas, cadenas, sucursales.
4. Cómo filtrar por zona y cómo identificar un producto entre cadenas.
5. Qué tan completas vienen las promociones.
6. Qué problemas de calidad aparecen.
7. Cuánto ocuparía cada día según el filtro que elijamos.

Las conclusiones están resumidas en `docs/fuente_sepa.md`.

In [1]:
import json
import urllib.request
import zipfile
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CRUDO = RAIZ / "data" / "crudo"
CRUDO.mkdir(parents=True, exist_ok=True)

DIA_SEMANA = "lunes"  # el archivo que se analiza
ID_DATASET = "sepa-precios"
API_CKAN = "https://datos.produccion.gob.ar/api/3/action/package_show?id=" + ID_DATASET

con = duckdb.connect()

def sql(consulta):
    return con.sql(consulta).df()

## 1. Archivos publicados y frecuencia de actualización

El dataset de datos.gob.ar es un espejo del portal de origen, **datos.produccion.gob.ar**. El espejo
mostraba fechas de modificación de julio de 2026, mientras que el origen está al día, así que se usa siempre el origen.

Consultamos la API CKAN del portal de origen para ver los recursos.

In [2]:
pedido = urllib.request.Request(API_CKAN, headers={"User-Agent": "Mozilla/5.0"})
paquete = json.load(urllib.request.urlopen(pedido))["result"]

recursos = pd.DataFrame([
    {
        "nombre": r["name"],
        "formato": r["format"],
        "ultima_modificacion": r.get("last_modified"),
        "tamano_mb": round((r.get("size") or 0) / 1e6, 1),
        "url": r["url"],
    }
    for r in paquete["resources"]
])
print("Periodicidad declarada:", paquete.get("accrualPeriodicity"))
recursos[["nombre", "formato", "ultima_modificacion", "tamano_mb"]]

Periodicidad declarada: None


,nombre,formato,ultima_modificacion,tamano_mb
0,Viernes,ZIP,2026-11-13T16:19:31.534246,335.8
1,Jueves,ZIP,2026-11-12T16:19:19.797725,328.9
2,Miércoles,ZIP,2026-11-11T16:19:05.704533,314.2
3,Martes,ZIP,2026-11-10T16:18:39.833283,314.1
4,Lunes,ZIP,2026-11-16T16:20:11.620780,335.1
5,Domingo,ZIP,2026-11-15T16:19:16.437506,330.5
6,Sábado,ZIP,2026-11-14T16:19:16.993342,336.2
7,Traductor de código provinciales,XLSX,2024-08-20T17:24:22.110823,0.0
8,Metadata (con detalle de los campos),PDF,2024-08-20T16:54:55.573247,0.3


**Lectura:** hay **un zip por día de la semana** (`sepa_lunes.zip` … `sepa_domingo.zip`) en URLs fijas.
Cada día se pisa el archivo de ese día de la semana, así que el portal guarda **una ventana móvil de 7 días**.
Cada zip pesa unos 310-340 MB y se actualiza alrededor de las **16:20 UTC (13:20 hora argentina)**.

Esto implica que:
- Si un día falla la descarga, hay hasta 6 días para recuperarlo antes de que se pise.
- Al arrancar el pipeline se pueden bajar los últimos 7 días de una vez.
- Hay que verificar la fecha **adentro** del zip: antes de las 13:20 de un lunes, `sepa_lunes.zip` todavía tiene los datos del lunes anterior.

## 2. Descarga y estructura del paquete

In [3]:
url = recursos.loc[recursos["nombre"].str.lower() == DIA_SEMANA, "url"].iloc[0]
zip_externo = CRUDO / f"sepa_{DIA_SEMANA}.zip"

if not zip_externo.exists():
    pedido = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(pedido) as r, open(zip_externo, "wb") as f:
        while bloque := r.read(1 << 20):
            f.write(bloque)

print(f"{zip_externo.name}: {zip_externo.stat().st_size / 1e6:.0f} MB")

with zipfile.ZipFile(zip_externo) as z:
    internos = pd.DataFrame(
        [{"archivo": i.filename, "mb": round(i.file_size / 1e6, 2), "fecha_zip": pd.Timestamp(*i.date_time)}
         for i in z.infolist() if not i.is_dir()]
    )
internos["id_comercio"] = internos["archivo"].str.extract(r"comercio-sepa-(\d+)_").astype(int)
internos.sort_values("mb", ascending=False)

sepa_lunes.zip: 335 MB


,archivo,mb,fecha_zip,id_comercio
16,2026-11-16/sepa_2_comercio-sepa-10_2026-11-16_01-05-08.zip,126.82,2026-11-16 10:05:22,10
7,2026-11-16/sepa_2_comercio-sepa-15_2026-11-16_01-05-08.zip,98.30,2026-11-16 10:06:40,15
12,2026-11-16/sepa_2_comercio-sepa-11_2026-11-16_01-05-08.zip,43.78,2026-11-16 10:06:54,11
13,2026-11-16/sepa_2_comercio-sepa-2_2026-11-16_01-05-08.zip,23.16,2026-11-16 10:05:10,2
11,2026-11-16/sepa_1_comercio-sepa-13_2026-11-16_09-05-10.zip,16.99,2026-11-16 06:07:52,13
4,2026-11-16/sepa_1_comercio-sepa-12_2026-11-16_09-05-10.zip,8.45,2026-11-16 06:06:26,12
0,2026-11-16/sepa_1_comercio-sepa-24_2026-11-16_09-05-10.zip,7.38,2026-11-16 06:06:22,24
5,2026-11-16/sepa_2_comercio-sepa-9_2026-11-16_01-05-08.zip,7.00,2026-11-16 10:06:46,9
2,2026-11-16/sepa_1_comercio-sepa-21_2026-11-16_09-05-10.zip,1.05,2026-11-16 06:07:04,21
15,2026-11-16/sepa_1_comercio-sepa-8_2026-11-16_09-05-10.zip,0.90,2026-11-16 06:06:36,8


El zip externo trae una carpeta con la fecha (`AAAA-MM-DD/`) y adentro **un zip por comercio**
(`sepa_N_comercio-sepa-{id_comercio}_{fecha}_{hora}.zip`). Cada zip interno trae los tres archivos de la
especificación: `comercio.csv`, `sucursales.csv` y `productos.csv`.

In [4]:
fecha_paquete = internos["archivo"].str.split("/").str[0].iloc[0]
DESTINO = CRUDO / fecha_paquete
problemas_zip = []

if not DESTINO.exists():
    with zipfile.ZipFile(zip_externo) as z:
        for nombre in internos["archivo"]:
            carpeta = DESTINO / Path(nombre).stem
            try:
                with z.open(nombre) as interno, zipfile.ZipFile(interno) as zi:
                    zi.extractall(carpeta)
            except zipfile.BadZipFile as e:
                problemas_zip.append((nombre, str(e)))

with zipfile.ZipFile(zip_externo) as z:
    for nombre in internos["archivo"]:
        with z.open(nombre) as interno:
            try:
                with zipfile.ZipFile(interno) as zi:
                    fechas = {pd.Timestamp(*i.date_time).date() for i in zi.infolist()}
                    internos.loc[internos["archivo"] == nombre, "fecha_csv"] = str(max(fechas))
            except zipfile.BadZipFile:
                internos.loc[internos["archivo"] == nombre, "fecha_csv"] = "ZIP ROTO"

print("Fecha del paquete:", fecha_paquete)
internos[internos["fecha_csv"] != fecha_paquete][["archivo", "mb", "fecha_csv"]]

Fecha del paquete: 2026-11-16


,archivo,mb,fecha_csv
3,2026-11-16/sepa_1_comercio-sepa-4_2026-11-16_09-05-10.zip,0.01,2026-11-15
6,2026-11-16/sepa_1_comercio-sepa-47_2026-11-16_09-05-10.zip,0.49,2025-06-11
8,2026-11-16/sepa_1_comercio-sepa-3_2026-11-16_09-05-10.zip,0.07,2026-11-15
9,2026-11-16/sepa_2_comercio-sepa-36_2026-11-16_01-05-08.zip,0.00,ZIP ROTO
15,2026-11-16/sepa_1_comercio-sepa-8_2026-11-16_09-05-10.zip,0.90,2026-11-15


**Problemas en el paquete:** un comercio viene con el **zip vacío (0 bytes)** y otro trae CSVs con
**fecha de hace más de un año**: lo vuelven a publicar todos los días sin actualizar. El pipeline tiene que
detectar ambas cosas y no tomar como vigentes precios viejos.

### Formato de los CSV

In [5]:
ejemplo = next(DESTINO.glob("*comercio-sepa-12_*"))
for archivo in ["comercio.csv", "sucursales.csv", "productos.csv"]:
    crudo = (ejemplo / archivo).read_bytes()
    lineas = crudo.decode("utf-8-sig").split("\r\n")
    print(f"--- {archivo}  (BOM UTF-8: {crudo[:3] == b'\xef\xbb\xbf'}, fin de línea CRLF: {b'\r\n' in crudo[:2000]})")
    print("\n".join(lineas[:2]))
    print("...")
    print(repr(lineas[-3:]))

--- comercio.csv  (BOM UTF-8: False, fin de línea CRLF: True)
id_comercio|id_bandera|comercio_cuit|comercio_razon_social|comercio_bandera_nombre|comercio_bandera_url|comercio_ultima_actualizacion|comercio_version_sepa
12|1|30548083156|COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.|COTO CICSA|http://www.coto.com.ar/|2026-11-16T01:11:59-03:00|1.0
...
['', 'Ultima actualizacion: 2026-11-16T01:11:59-03:00', '']
--- sucursales.csv  (BOM UTF-8: False, fin de línea CRLF: True)
id_comercio|id_bandera|id_sucursal|sucursales_nombre|sucursales_tipo|sucursales_calle|sucursales_numero|sucursales_latitud|sucursales_longitud|sucursales_observaciones|sucursales_barrio|sucursales_codigo_postal|sucursales_localidad|sucursales_provincia|sucursales_lunes_horario_atencion|sucursales_martes_horario_atencion|sucursales_miercoles_horario_atencion|sucursales_jueves_horario_atencion|sucursales_viernes_horario_atencion|sucursales_sabado_horario_atencion|sucursales_domingo_horario_atencion
12|1|2|AV.SAN MARTIN|Sup

--- productos.csv  (BOM UTF-8: False, fin de línea CRLF: True)
id_comercio|id_bandera|id_sucursal|id_producto|productos_ean|productos_descripcion|productos_cantidad_presentacion|productos_unidad_medida_presentacion|productos_marca|productos_precio_lista|productos_precio_referencia|productos_cantidad_referencia|productos_unidad_medida_referencia|productos_precio_unitario_promo1|productos_leyenda_promo1|productos_precio_unitario_promo2|productos_leyenda_promo2
12|1|171|7792070072737|1|ACEITE DE OLIVA VIRGEN NUCETE BID 2 LTR|2|ltr|NUCETE|46957|23478.5|1|ltr||||
...
['', 'Ultima actualizacion: 2026-11-16T01:11:59-03:00', '']


- Separador **`|`**, codificación **UTF-8** (algunos con BOM), fin de línea **CRLF**, con encabezado.
- Al final de cada archivo hay **una línea en blanco y un pie** `Ultima actualizacion: ...` que no es un dato
  y hay que descartarlo al leer.
- Se lee todo como texto (`all_varchar`) y se filtran las filas cuyo `id_comercio` no es numérico:
  así se descarta el pie sin depender de que todos los comercios lo escriban igual.

In [6]:
patron = str(DESTINO / "*" / "{}.csv").replace("\\", "/")
for tabla in ["comercio", "sucursales", "productos"]:
    con.execute(f'''
        create or replace table {tabla} as
        select * exclude (filename), regexp_extract(filename, 'comercio-sepa-(\\d+)_', 1) as id_comercio_archivo
        from read_csv('{patron.format(tabla)}', delim='|', header=true, all_varchar=true,
                      null_padding=true, filename=true)
        where regexp_full_match(id_comercio, '[0-9]+')
    ''')

tam_descomprimido = sum(f.stat().st_size for f in DESTINO.rglob("*.csv")) / 1e9
print(f"Descomprimido: {tam_descomprimido:.2f} GB de CSV")
sql('''
    select 'comercio' as tabla, count(*) as filas from comercio
    union all select 'sucursales', count(*) from sucursales
    union all select 'productos', count(*) from productos
''')

Descomprimido: 1.68 GB de CSV


,tabla,filas
0,comercio,29
1,sucursales,3016
2,productos,14917752


### Columnas de cada tabla

In [7]:
pd.concat([
    sql(f"select '{t}' as tabla, column_name as columna from (describe {t}) where column_name <> 'id_comercio_archivo'")
    for t in ["comercio", "sucursales", "productos"]
]).groupby("tabla")["columna"].apply(lambda c: ", ".join(c)).to_frame()

,columna
tabla,
comercio,"id_comercio, id_bandera, comercio_cuit, comercio_razon_social, comercio_band..."
productos,"id_comercio, id_bandera, id_sucursal, id_producto, productos_ean, productos_..."
sucursales,"id_comercio, id_bandera, id_sucursal, sucursales_nombre, sucursales_tipo, su..."


Todas las columnas coinciden con la especificación oficial (Anexo II, Disposición SEPA, guardado en
`data/crudo/especificacion_sepa.pdf`). Los campos clave:

- **`id_comercio` + `id_bandera` + `id_sucursal`** identifican una sucursal. `id_sucursal` solo es único dentro del comercio.
- **`id_producto`** es el código **EAN/UPC** cuando `productos_ean = 1`; si vale 0 es un código interno del comercio.
- **`productos_precio_lista`** es el precio sin ningún descuento. Es el que vamos a usar como precio principal.
- **`productos_precio_unitario_promo1`** es la promoción de alcance general y **`promo2`** la promoción con condiciones (tarjetas, medios de pago, etc.).

## 3. Comercios y banderas

In [8]:
sql('''
    select c.id_comercio::int as id_comercio, c.id_bandera::int as id_bandera, comercio_bandera_nombre,
           comercio_ultima_actualizacion,
           count(distinct s.id_sucursal) as sucursales
    from comercio c
    left join sucursales s using (id_comercio, id_bandera)
    group by all
    order by 1, 2
''')

,id_comercio,id_bandera,comercio_bandera_nombre,comercio_ultima_actualizacion,sucursales
0,2,1,La Anonima,2026-11-16T04:00:02-03:00,185
1,2,2,Topsy,2026-11-16T04:00:02-03:00,0
2,2,3,Bomba,2026-11-16T04:00:02-03:00,0
3,3,1,DEHEZA S.A.I.C.F. e I.,2026-11-15T16:00:02-03:00,41
4,4,1,ESTACION LIMA S.A.,2026-11-15T16:00:03-03:00,1
5,5,1,California Supermercados,2026-11-16T09:00:02-03:00,7
6,6,1,Supermercados Comodin,2026-11-16T09:30:00-03:00,23
7,6,2,Maxi Comodin,2026-11-16T09:30:00-03:00,3
8,8,1,Mariano Max,2026-11-15T16:30:35-03:00,11
9,9,1,Vea,2026-11-16T07:01:55-03:00,154


Son **16 comercios (empresas) y 29 banderas** con datos. Incluye las grandes cadenas (Carrefour, Cencosud con Vea,
Disco y Jumbo, Coto, DIA, Changomas, La Anónima, Cooperativa Obrera), algunas regionales, y también comercios
que no son supermercados (Farmacity/Simplicity, la tienda de Axion).

`comercio_ultima_actualizacion` **no es confiable**: Changomas declara 2017 y 2021 aunque sus precios
son del día. La fecha válida es la de la carpeta del zip.

In [9]:
sql('''
    select id_comercio::int as id_comercio, count(*) as filas,
           count(distinct id_sucursal) as sucursales,
           count(distinct id_producto) as productos
    from productos group by 1 order by filas desc
''')

,id_comercio,filas,sucursales,productos
0,15,4336175,970,4508
1,10,3438442,712,36808
2,11,2575812,93,27742
3,9,1262349,268,11969
4,12,1041157,120,14701
5,2,1012894,185,9426
6,13,709486,156,10627
7,24,383767,334,1950
8,21,43323,25,4875
9,8,42035,11,4714


## 4. Zona: cómo filtrar por ubicación

In [10]:
provincias = pd.read_excel(CRUDO / "maestro-provincias.xlsx") if (CRUDO / "maestro-provincias.xlsx").exists() else None
sql('''
    select coalesce(sucursales_provincia, '(vacío)') as provincia, count(*) as sucursales
    from sucursales group by 1 order by 2 desc
''').head(10)

,provincia,sucursales
0,AR-C,1160
1,AR-B,960
2,AR-X,175
3,AR-E,89
4,AR-M,73
5,AR-R,66
6,(vacío),61
7,AR-Q,60
8,AR-A,51
9,AR-S,51


In [11]:
sql('''
    select sucursales_localidad, count(*) as sucursales
    from sucursales where sucursales_provincia = 'AR-B'
    group by 1 order by 2 desc limit 12
''')

,sucursales_localidad,sucursales
0,BUENOS AIRES,418
1,Bahía Blanca,30
2,Mar Del Plata,23
3,LA PLATA,10
4,MAR DEL PLATA,9
5,Mar del Plata,8
6,Quilmes,8
7,Pilar,7
8,San Isidro,7
9,Lomas de Zamora,6


- La provincia viene en código **ISO 3166-2** (`AR-C` = CABA, `AR-B` = Buenos Aires). Hay 61 sucursales sin
  provincia y una que dice "Buenos Aires" en texto.
- **La localidad no sirve para filtrar**: es texto libre, con mayúsculas y acentos mezclados ("LANUS", "Lanús"),
  y una cadena pone "BUENOS AIRES" como localidad en 418 sucursales de la provincia.
- **Latitud y longitud están completas en todas las sucursales.** Son la forma más confiable de definir la zona.

Para definir **CABA + GBA** usamos la distancia al Obelisco. El conurbano (los 24 partidos)
queda aproximadamente dentro de los 40 km.

In [12]:
con.execute('''
    create or replace table sucursales_geo as
    select *,
        try_cast(sucursales_latitud as double) as lat,
        try_cast(sucursales_longitud as double) as lon,
        2 * 6371 * asin(sqrt(
            pow(sin(radians(try_cast(sucursales_latitud as double) - (-34.6037)) / 2), 2) +
            cos(radians(-34.6037)) * cos(radians(try_cast(sucursales_latitud as double))) *
            pow(sin(radians(try_cast(sucursales_longitud as double) - (-58.3816)) / 2), 2)
        )) as km_obelisco
    from sucursales
''')
sql('''
    select
        count(*) filter (where sucursales_provincia = 'AR-C') as caba_por_provincia,
        count(*) filter (where km_obelisco <= 15) as hasta_15km,
        count(*) filter (where km_obelisco <= 30) as hasta_30km,
        count(*) filter (where km_obelisco <= 40) as hasta_40km,
        count(*) filter (where km_obelisco <= 50) as hasta_50km,
        count(*) filter (where sucursales_provincia = 'AR-C' and km_obelisco > 20) as caba_lejos_raro,
        count(*) filter (where lat not between -56 and -21 or lon not between -74 and -53) as fuera_de_argentina
    from sucursales_geo
''')

,caba_por_provincia,hasta_15km,hasta_30km,hasta_40km,hasta_50km,caba_lejos_raro,fuera_de_argentina
0,1160,1234,1598,1699,1743,33,1


In [13]:
sql('''
    select id_comercio, comercio_bandera_nombre, count(*) as sucursales_amba
    from sucursales_geo join comercio using (id_comercio, id_bandera)
    where km_obelisco <= 40
    group by all order by 3 desc
''')

,id_comercio,comercio_bandera_nombre,sucursales_amba
0,15,Supermercados DIA,711
1,10,Express,407
2,24,FARMACITY,183
3,12,COTO CICSA,104
4,10,Market,43
5,23,Axion Energy,36
6,9,Disco,36
7,3,DEHEZA S.A.I.C.F. e I.,33
8,24,SIMPLICITY,32
9,10,Hipermercado Carrefour,32


## 5. Productos: identificación entre cadenas

In [14]:
sql('''
    select productos_ean, count(*) as filas, count(distinct id_producto) as productos,
           round(100 * count(*) / sum(count(*)) over (), 1) as pct_filas
    from productos group by 1
''')

,productos_ean,filas,productos,pct_filas
0,0,168956,3761,1.1
1,1,14748796,77706,98.9


In [15]:
sql('''
    with por_producto as (
        select id_producto, count(distinct id_comercio) as comercios
        from productos where productos_ean = '1' group by 1
    )
    select comercios as presente_en_n_comercios, count(*) as productos,
           sum(count(*)) over (order by comercios desc) as acumulado
    from por_producto group by 1 order by 1 desc
''')

,presente_en_n_comercios,productos,acumulado
0,13,3,3.0
1,12,9,12.0
2,11,52,64.0
3,10,140,204.0
4,9,367,571.0
5,8,692,1263.0
6,7,1338,2601.0
7,6,1587,4188.0
8,5,1930,6118.0
9,4,2515,8633.0


- El **99% de las filas usa EAN** y `id_producto` siempre tiene 13 dígitos. El EAN es el identificador común
  entre cadenas: el mismo código en Coto y en Carrefour es el mismo producto.
- La mayoría de los productos (57 mil) aparecen en un solo comercio, pero **unos 2.600 están en 7 o más
  comercios** y alcanzan para armar una canasta comparable.
- La descripción y la marca son texto libre de cada cadena (por ejemplo, DIA abrevia "LA SE", "MANFR"),
  así que el nombre del producto se toma de la versión más frecuente o de la cadena con mejor descripción.

In [16]:
sql('''
    select id_producto, any_value(productos_descripcion) as descripcion_ejemplo,
           count(distinct id_comercio) as comercios, count(*) as sucursales,
           median(productos_precio_lista::double) as precio_mediana,
           min(productos_precio_lista::double) as minimo, max(productos_precio_lista::double) as maximo
    from productos
    where productos_ean = '1'
    group by 1
    order by comercios desc, sucursales desc
    limit 20
''')

,id_producto,descripcion_ejemplo,comercios,sucursales,precio_mediana,minimo,maximo
0,7791337007253,CHOCOLATADA CINDOR,13,2751,8535.00,3592.0,11500.0
1,7790580716707,GALL SALADIX JAMON,13,2420,1901.00,875.0,3200.0
2,7790895000997,COCA COLA,13,2180,6100.00,4081.0,9450.0
3,7791337007260,CHOCOLATADA CINDOR,12,2724,3050.00,1585.0,3800.0
4,7790580199913,GOMITAS MOGUL OSITOS,12,2538,990.00,699.0,1240.0
5,7790010002660,TOA SL ESPECIAL,12,2525,4266.99,2609.0,5050.0
6,7790895641183,POWERADE FRUTAS TROPICAL,12,2515,2200.00,1670.0,3600.0
7,7790072002080,SAL FINA CELUSAL CAJA,12,2474,2026.00,1329.0,2350.0
8,7790895640025,POWERADE MOUNTAIN BLAST,12,2471,2200.00,1670.0,3600.0
9,7790895647017,JUGO NATURAL MANZANA ZINC CEPITA X 1 LT,12,2291,3450.00,2181.0,5600.0


## 6. Promociones

In [17]:
sql('''
    select id_comercio::int as id_comercio,
        round(100 * avg((nullif(productos_precio_unitario_promo1, '') is not null)::int), 1) as pct_promo1,
        round(100 * avg((nullif(productos_precio_unitario_promo2, '') is not null)::int), 1) as pct_promo2
    from productos group by 1 order by pct_promo1 desc
''')

,id_comercio,pct_promo1,pct_promo2
0,15,22.5,11.9
1,13,16.6,0.0
2,10,4.9,0.0
3,2,2.1,0.0
4,21,2.0,0.0
5,12,0.8,0.0
6,3,0.0,0.0
7,6,0.0,0.0
8,24,0.0,0.0
9,4,0.0,0.0


In [18]:
sql('''
    select coalesce(productos_leyenda_promo1, productos_leyenda_promo2) as leyenda, count(*) as filas
    from productos
    where nullif(productos_leyenda_promo1, '') is not null or nullif(productos_leyenda_promo2, '') is not null
    group by 1 order by 2 desc limit 10
''')

,leyenda,filas
0,25% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,187099
1,Promo A valida desde el 12/11/2026 hasta 17/11/2026,74212
2,35% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,63931
3,15% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,63350
4,20% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,61407
5,30% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,59999
6,26% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,40671
7,35% de descuento. 35% ELVIVE - Vigencia: Desde el 11/11/2026 hasta el 16/11/...,33647
8,40% de descuento. Precio Promocional Exclusivo DIA en Sucursales Indicadas H...,32248
9,35% de descuento. 35% SEDAL - Vigencia: Desde el 11/11/2026 hasta el 16/11/2026,30865


- Solo el **8,7% de las filas trae promo1** y el **3,4% trae promo2**, y casi todo viene de DIA (22,5% y 11,9%)
  y de la Cooperativa Obrera (16,6%). **La mitad de los comercios nunca informa promociones**, aunque las tengan.
- Las leyendas son texto libre, algunas con fecha de vigencia y otras no.
- La pregunta 5 del análisis ("¿las promociones son reales?") solo se puede responder para DIA y la Cooperativa
  Obrera, y para el resto queda como limitación.

## 7. Calidad de los datos

In [19]:
sql('''
    select 'precio_lista no numérico' as control, count(*) as filas from productos where try_cast(productos_precio_lista as double) is null
    union all select 'precio_lista <= 0', count(*) from productos where try_cast(productos_precio_lista as double) <= 0
    union all select 'precio_lista > 5 millones', count(*) from productos where try_cast(productos_precio_lista as double) > 5e6
    union all select 'duplicados (sucursal + producto)', count(*) from (
        select id_comercio, id_bandera, id_sucursal, id_producto from productos group by all having count(*) > 1)
    union all select 'precios de sucursales que no están en sucursales.csv', count(*) from productos p
        anti join sucursales s using (id_comercio, id_bandera, id_sucursal)
    union all select 'sucursales sin ningún precio', count(*) from sucursales s
        anti join (select distinct id_comercio, id_bandera, id_sucursal from productos) p using (id_comercio, id_bandera, id_sucursal)
''')

,control,filas
0,precio_lista no numérico,29
1,precio_lista <= 0,70
2,precio_lista > 5 millones,332
3,duplicados (sucursal + producto),49
4,precios de sucursales que no están en sucursales.csv,55329
5,sucursales sin ningún precio,30


In [20]:
sql('''
    select productos_precio_lista, count(*) as filas from productos
    where try_cast(productos_precio_lista as double) is null group by 1
''')

,productos_precio_lista,filas
0,None,29


In [21]:
sql('''
    select productos_unidad_medida_presentacion as unidad, count(*) as filas
    from productos group by 1 order by 2 desc limit 25
''').T

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
unidad,UNI,GR,EA,ML,gr,KG,gr.,cm3,grm,ml,...,cu,un.,uni,pck,kgm,cmq,KGM,cc.,Kg,uni.
filas,3373280,2234993,1620225,1115373,986446,834403,655977,532518,500837,464004,...,137110,117862,109667,87015,77186,70891,61695,30928,25408,24299


In [22]:
sql("select sucursales_tipo, count(*) as sucursales from sucursales group by 1 order by 2 desc")

,sucursales_tipo,sucursales
0,Autoservicio,1701
1,Supermercado,957
2,Hipermercado,336
3,supermercado,19
4,Web,2
5,autoservicio,1


Problemas para que la etapa 2 controle o la etapa 4 limpie:
- Precios en cero y vacíos. Los precios de más de 5 millones (332 filas) son en su mayoría electrónica (televisores),
  así que el control de "precio absurdo" tiene que comparar contra la mediana del producto y no usar un tope fijo.
- Filas duplicadas para la misma sucursal y producto.
- 55 mil filas de precios de 14 sucursales que no aparecen en `sucursales.csv`, y 30 sucursales sin precios.
- 33 sucursales de CABA están a más de 20 km del Obelisco (coordenadas mal cargadas) y una cae fuera de Argentina.
- Cuatro comercios publican el paquete la tarde anterior (fecha del CSV = día anterior). Es normal: son los precios que rigen el día del paquete.
- Unidades de medida sin normalizar ("GR", "gr", "gr.", "grm"…), y lo mismo pasa con el tipo de sucursal ("Supermercado" / "supermercado").

## 8. Tamaño según el filtro

Guardamos cada escenario como Parquet comprimido (zstd) y medimos cuánto ocuparía **por día**.
La tabla de precios se guarda **normalizada**: solo ids, precios y promociones. La descripción, la marca y la
presentación van a una tabla de productos aparte que cambia poco.

In [23]:
import tempfile

con.execute('''
    create or replace table precios as
    select p.id_comercio::int as id_comercio, p.id_bandera::int as id_bandera, p.id_sucursal,
           p.id_producto, p.productos_ean = '1' as es_ean,
           try_cast(p.productos_precio_lista as decimal(12,2)) as precio_lista,
           try_cast(p.productos_precio_unitario_promo1 as decimal(12,2)) as precio_promo1,
           nullif(p.productos_leyenda_promo1, '') as leyenda_promo1,
           try_cast(p.productos_precio_unitario_promo2 as decimal(12,2)) as precio_promo2,
           nullif(p.productos_leyenda_promo2, '') as leyenda_promo2,
           s.sucursales_provincia as provincia, s.km_obelisco
    from productos p
    left join sucursales_geo s using (id_comercio, id_bandera, id_sucursal)
''')
con.execute('''
    create or replace table productos_frecuentes as
    select id_producto from precios where es_ean group by 1 having count(distinct id_comercio) >= 3
''')

escenarios = {
    "Todo el país": "true",
    "Solo CABA": "provincia = 'AR-C'",
    "CABA + GBA (40 km)": "km_obelisco <= 40",
    "CABA + GBA, productos EAN en 3+ comercios": "km_obelisco <= 40 and id_producto in (select id_producto from productos_frecuentes)",
}
carpeta = Path(tempfile.mkdtemp())
filas = []
for nombre, filtro in escenarios.items():
    destino = (carpeta / f"{len(filas)}.parquet").as_posix()
    con.execute(f'''copy (select * exclude (provincia, km_obelisco) from precios where {filtro}
                         order by id_comercio, id_bandera, id_sucursal, id_producto)
                    to '{destino}' (format parquet, compression zstd)''')
    n, sucs, prods = con.execute(f"select count(*), count(distinct (id_comercio, id_bandera, id_sucursal)), count(distinct id_producto) from precios where {filtro}").fetchall()[0]
    mb = Path(destino).stat().st_size / 1e6
    filas.append({"escenario": nombre, "filas": n, "sucursales": sucs, "productos": prods,
                  "mb_por_dia": round(mb, 1), "gb_por_año": round(mb * 365 / 1000, 2)})

destino = (carpeta / "dim_productos.parquet").as_posix()
con.execute(f'''copy (select id_producto, mode(productos_descripcion) as descripcion, mode(productos_marca) as marca,
                             mode(productos_cantidad_presentacion) as cantidad, mode(productos_unidad_medida_presentacion) as unidad
                      from productos group by 1) to '{destino}' (format parquet, compression zstd)''')
print(f"Tabla de productos (aparte, se actualiza poco): {Path(destino).stat().st_size / 1e6:.1f} MB")

# Versión agregada: una fila por bandera, producto y zona (CABA / GBA) con mediana y rango
destino = (carpeta / "agregado.parquet").as_posix()
con.execute(f'''copy (
    select id_comercio, id_bandera, id_producto,
           case when provincia = 'AR-C' then 'CABA' else 'GBA' end as zona,
           count(*) as sucursales, median(precio_lista) as precio_mediana,
           min(precio_lista) as precio_min, max(precio_lista) as precio_max,
           median(precio_promo1) as promo1_mediana, median(precio_promo2) as promo2_mediana
    from precios where km_obelisco <= 40
    group by all order by all
) to '{destino}' (format parquet, compression zstd)''')
n = con.execute(f"select count(*) from '{destino}'").fetchall()[0][0]
mb = Path(destino).stat().st_size / 1e6
filas.append({"escenario": "CABA + GBA agregado por bandera, producto y zona", "filas": n, "sucursales": None,
              "productos": None, "mb_por_dia": round(mb, 1), "gb_por_año": round(mb * 365 / 1000, 2)})
tamanos = pd.DataFrame(filas)
tamanos

Tabla de productos (aparte, se actualiza poco): 1.8 MB


,escenario,filas,sucursales,productos,mb_por_dia,gb_por_año
0,Todo el país,14917752,3000.0,81455.0,34.8,12.70
1,Solo CABA,4220236,1150.0,59584.0,8.1,2.96
2,CABA + GBA (40 km),7458991,1683.0,66158.0,15.2,5.54
3,"CABA + GBA, productos EAN en 3+ comercios",4534686,1674.0,11837.0,7.0,2.54
4,"CABA + GBA agregado por bandera, producto y zona",360229,NaN,NaN,2.8,1.01


**Lectura:** guardar el detalle por sucursal de CABA + GBA ocupa unos 15 MB por día, unos **5,5 GB por año**.
Eso no entra en un repo de git (GitHub recomienda repos de menos de 1 GB y cada commit de datos queda para
siempre en el historial). Recortar productos no alcanza: la versión más chica pasa los 2,5 GB por año, y
hasta la versión **agregada** por bandera, producto y zona llega a 1 GB por año.

Conclusión: **los datos diarios no van en git**. Van como archivos de un Release de GitHub, y en el repo
quedan solo las tablas chicas que usan la app y el reporte.

In [24]:
sql('''
    select id_comercio, count(*) as filas, min(precio_lista) as minimo, max(precio_lista) as maximo,
           any_value(id_producto) as producto_ejemplo
    from precios where precio_lista > 5e6 group by 1 order by 2 desc
''')

,id_comercio,filas,minimo,maximo,producto_ejemplo
0,10,254,5199000.0,12999000.0,3617703007242
1,12,67,5299999.0,11399999.0,7796962001898
2,13,8,6449999.0,6449999.0,0199272245193
3,2,3,5606999.0,6499999.0,7790653062267


## Conclusiones y propuesta de filtro

Ver `docs/fuente_sepa.md` y `docs/decisiones.md` para la versión completa. En resumen:

- **Zona: CABA + GBA**, definida como las sucursales a **40 km o menos del Obelisco** (por coordenadas,
  porque la localidad no es confiable). Es la zona con más sucursales y donde compiten todas las cadenas grandes,
  así que permite comparar cadenas dentro de una misma zona.
- **Productos: todos los de esa zona.** Así queda abierta la elección de la canasta y de la prima de marca para la etapa 4.
- **Almacenamiento en dos niveles**, porque los datos diarios no entran en git:
  - El **detalle por sucursal** (~15 MB por día) se guarda fuera del repo, como archivo adjunto de un Release de GitHub (un archivo por día).
  - En el repo van solo las **tablas chicas** que usan la app y el reporte (canasta, índices, métricas), que se regeneran en cada corrida.